# Previsão de Pontuação na Fórmula 1 (Top 10)

Prever se um piloto termina uma corrida entre os 10 primeiros (pontua), usando os dados históricos da Fórmula 1 de 1950 a 2024.

Tarefa: classificação binária. Alvo: `pontuou` (1 se o piloto terminou entre os 10 primeiros, 0 caso contrário).

Cada seção indica o responsável. Toda figura ou tabela precisa vir com uma interpretação escrita logo abaixo.

## 1. Identificação e descrição do problema

Responsável: Gabriel (Pessoa A)

Título: Previsão de Pontuação na Fórmula 1 (Top 10)

Integrantes:

| Papel | Nome completo | GitHub |
|---|---|---|
| Pessoa A | Gabriel Rodrigues dos Santos | @Gabriell-Rodrigues |
| Pessoa B | [preencher] | [@usuario] |
| Pessoa C | [preencher] | [@usuario] |

Fonte dos dados: Formula 1 World Championship (1950 a 2024), do Kaggle (rohanrao). Os arquivos ficam versionados na pasta `data/` do repositório e são lidos pela URL pública (raw do GitHub), sem depender de nenhum arquivo local.

Objetivo: prever se um piloto termina uma corrida entre os 10 primeiros usando apenas informações conhecidas antes ou no início da prova, como a posição de largada, a equipe, o piloto e o circuito. Funciona como uma estimativa do desempenho esperado antes da corrida acontecer.

Atributo-alvo: `pontuou`. Vale 1 quando o piloto termina entre os 10 primeiros da ordem final e 0 caso contrário. A regra de pontuação da F1 mudou várias vezes desde 1950 (já premiou os 5, 6 ou 8 primeiros), então o alvo é definido pela posição final (top 10), que é um critério estável em todo o período.

Atributos preditivos candidatos:

- `grid`: posição de largada
- `constructorId`: equipe
- `driverId`: piloto
- `circuitId`: circuito
- `year` e `round`: ano e etapa da temporada

Não entram como preditor as colunas que só existem depois da corrida (posição final, pontos, voltas, tempo, volta mais rápida e status). Elas servem apenas para montar o alvo, para não haver vazamento de informação.

Tipo da tarefa: classificação binária. O alvo assume duas categorias (pontuou ou não pontuou), por isso é um problema de classificação, e não de regressão, que seria o caso se o alvo fosse um número contínuo.

## Preparação: carregamento dos dados

Responsável: Gabriel (Pessoa A)

Carregar os arquivos da pasta `data/` pela URL pública do repositório (raw do GitHub), sem depender de arquivo local. Arquivos usados: results, races, drivers, constructors, qualifying, status, circuits.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

base = "https://raw.githubusercontent.com/Gabriell-Rodrigues/Unidade-3---Projeto-Final-Aprendizado-de-M-quina/main/data/"

# os arquivos da F1 marcam ausência com \N, então convertemos esse marcador para NaN na leitura
results = pd.read_csv(base + "results.csv", na_values=r"\N")
races = pd.read_csv(base + "races.csv", na_values=r"\N")
drivers = pd.read_csv(base + "drivers.csv", na_values=r"\N")
constructors = pd.read_csv(base + "constructors.csv", na_values=r"\N")
qualifying = pd.read_csv(base + "qualifying.csv", na_values=r"\N")
status = pd.read_csv(base + "status.csv", na_values=r"\N")
circuits = pd.read_csv(base + "circuits.csv", na_values=r"\N")

results.shape

### Construção do atributo-alvo

Responsável: Gabriel (Pessoa A)

Cada linha de `results` é o resultado de um piloto em uma corrida. Juntamos essa tabela com `races` para trazer o ano, a rodada e o circuito, e criamos a coluna `pontuou` a partir de `positionOrder`: vale 1 quando a posição final é menor ou igual a 10 e 0 nos demais casos.

Usamos `positionOrder`, e não `position`, porque `position` fica vazia quando o piloto abandona a prova, enquanto `positionOrder` ordena todos os pilotos, inclusive os que não terminaram. As colunas que só existem depois da corrida são separadas em `pos_corrida` e nunca entram como preditor, para evitar vazamento.

In [ ]:
corridas = races[["raceId", "year", "round", "circuitId", "name"]]
df = results.merge(corridas, on="raceId", how="left")

df["pontuou"] = (df["positionOrder"] <= 10).astype(int)

pos_corrida = ["position", "positionText", "positionOrder", "points", "laps",
               "time", "milliseconds", "fastestLap", "rank",
               "fastestLapTime", "fastestLapSpeed", "statusId"]

preditores = ["grid", "constructorId", "driverId", "circuitId", "year", "round"]

df[preditores + ["pontuou"]].head()

## 2. Compreensão dos dados

Responsável: Gabriel (Pessoa A)

Apresentar e interpretar, sem se limitar a rodar os comandos:

- quantidade de registros e atributos
- tipos das variáveis
- valores ausentes
- duplicações
- inconsistências
- distribuição do atributo-alvo
- desbalanceamento entre as classes

In [ ]:
print("registros:", df.shape[0])
print("atributos:", df.shape[1])
print("periodo:", int(df["year"].min()), "a", int(df["year"].max()))
print("corridas:", df["raceId"].nunique())
print("pilotos:", df["driverId"].nunique())
print("equipes:", df["constructorId"].nunique())
print("circuitos:", df["circuitId"].nunique())
print()
df.dtypes.value_counts()

A base de análise reúne 26.759 resultados individuais (um piloto em uma corrida) entre 1950 e 2024, com 23 atributos, cobrindo 1.125 corridas, 861 pilotos, 211 equipes e 77 circuitos. Os tipos se dividem entre inteiros (identificadores e contagens), decimais e texto. Vários campos aparecem como decimais só porque têm valores ausentes, que o pandas representa como NaN e força a coluna a virar float; sem a ausência eles seriam inteiros. Vale notar que `driverId`, `constructorId` e `circuitId` são numéricos, mas representam categorias, não quantidades, então serão tratados como categóricos no pré-processamento.

In [ ]:
ausentes = df.isna().sum()
print("colunas com valores ausentes:")
print(ausentes[ausentes > 0].sort_values(ascending=False))
print()
print("ausentes nos preditores candidatos:")
print(df[preditores].isna().sum())

Os valores ausentes se concentram em colunas que só existem depois da corrida: `time` e `milliseconds` (19.079 cada), `fastestLap`, `fastestLapTime` e `fastestLapSpeed` (18.507), `rank` (18.249) e `position` (10.953). São campos de resultado que não vamos usar como preditor, então essa ausência não prejudica o modelo. Entre os preditores candidatos (grid, equipe, piloto, circuito, ano e etapa) não há nenhum valor ausente. A única falta fora das colunas de resultado é `number` (6 casos), o número do carro, que é opcional. Em resumo: o alvo e os preditores estão completos, e a ausência mora justamente no que será descartado para evitar vazamento.

In [ ]:
print("linhas totalmente duplicadas:", df.duplicated().sum())
print("resultId repetido:", results["resultId"].duplicated().sum())
print()
print("grid igual a 0 (largada do pit lane):", int((df["grid"] == 0).sum()))
print("grid minimo e maximo:", df["grid"].min(), df["grid"].max())
print()
print("positionOrder nulos:", int(df["positionOrder"].isna().sum()),
      "| minimo e maximo:", df["positionOrder"].min(), df["positionOrder"].max())
print("position (classificacao final) ausente:", int(df["position"].isna().sum()))
print("pontuou=1 sem classificacao final (abandono no top 10):",
      int(((df["pontuou"] == 1) & (df["position"].isna())).sum()))

Não há nenhuma linha totalmente duplicada nem `resultId` repetido, então cada registro é único e não é preciso remover duplicatas. Duas inconsistências merecem atenção. A primeira: `grid` igual a 0 aparece 1.638 vezes e não é uma posição real de largada, e sim a largada dos boxes (pit lane); é um valor especial que a Pessoa B pode tratar no pré-processamento. A segunda: `position` está ausente em 10.953 linhas, que são os pilotos que não terminaram a prova (abandono). Foi por isso que o alvo foi construído a partir de `positionOrder`, que não tem nenhum nulo e vai de 1 a 39, ordenando todos os pilotos. O efeito colateral é que 338 registros marcados como `pontuou=1` são pilotos que abandonaram mas ficaram no top 10 da ordem, o que acontece em corridas antigas com muitos abandonos; é um número pequeno diante das 26.759 linhas e não distorce o alvo.

In [ ]:
contagem = df["pontuou"].value_counts().sort_index()
proporcao = df["pontuou"].value_counts(normalize=True).sort_index()

print("contagem por classe:")
print(contagem)
print()
print("proporcao (%):")
print((proporcao * 100).round(2))

moderno = df[df["year"] >= 2010]
coerencia = (moderno["pontuou"] == (moderno["points"] > 0).astype(int)).mean()
print()
print("coerencia pontuou x points>0 na era 2010+:", round(coerencia * 100, 1), "%")

contagem.plot.bar(rot=0, grid=True)
plt.xticks([0, 1], ["nao pontuou", "pontuou"])
plt.ylabel("registros")
plt.show()

O alvo tem 15.438 registros na classe 0 (não pontuou), 57,7%, e 11.321 na classe 1 (pontuou), 42,3%. É um desbalanceamento leve, não severo: as duas classes estão bem representadas. Isso tem duas consequências para as próximas etapas. Primeiro, um modelo que sempre chutasse a classe majoritária (não pontuou) acertaria cerca de 57,7%, e é esse o baseline que a acurácia precisa superar. Segundo, como as classes não são iguais, a acurácia sozinha engana, então a avaliação da Seção 7 usa também precisão, revocação e F1, que medem o acerto na classe de quem pontua. Como checagem de coerência, na era moderna (2010 em diante, quando o 10º lugar dá ponto) o alvo `pontuou` coincide 100% com `points` maior que zero, o que confirma que a regra do top 10 reflete o sistema real de pontuação da categoria.

## 3. Análise exploratória

Responsável: [nome] (Pessoa B)

Usar, quando fizer sentido: histogramas, boxplots, gráficos de dispersão, tabelas de frequência, medidas de localidade e dispersão, correlações e a relação entre os preditores e o alvo.

Cada figura ou tabela precisa de uma interpretação escrita logo abaixo. Figura sem discussão não conta na avaliação.

In [ ]:
# Pessoa B
# gráficos e tabelas da análise exploratória

Interpretação:

## 4. Pré-processamento

Responsável: [nome] (Pessoa B)

Tratar o que for necessário e, para cada tratamento, explicar qual era o problema, o que foi feito e por quê.

Possíveis pontos: valores ausentes, variáveis categóricas, escalonamento, valores extremos, atributos irrelevantes, duplicações, inconsistências e classes desbalanceadas.

Montar as transformações com Pipeline e ColumnTransformer e ajustar apenas no treino, para não vazar informação do teste.

In [ ]:
# Pessoa B
# montar o pré-processamento das variáveis numéricas e categóricas

## 5. Separação dos dados

Responsável: [nome] (Pessoa B)

Separar treino e teste com estratificação pelo alvo. Justificar a proporção escolhida (por exemplo 80/20). Reservar o teste para o final e usar validação cruzada na comparação dos modelos.

Fazer a separação antes de ajustar as transformações.

In [ ]:
# Pessoa B
# separar treino e teste, estratificando pelo alvo

## 6. Modelagem

Responsável: [nome] (Pessoa C)

Modelos mínimos obrigatórios: SGDClassifier e RandomForestClassifier. Incluir também um baseline de referência.

Apresentar: o baseline, os modelos, os principais parâmetros, a comparação com validação cruzada e a escolha justificada do modelo final.

In [ ]:
# Pessoa C
# baseline de referência

In [ ]:
# Pessoa C
# treinar SGDClassifier e RandomForestClassifier e comparar com validação cruzada

## 7. Avaliação e discussão

Responsável: [nome] (Pessoa C)

Avaliar o modelo final no conjunto de teste com matriz de confusão, acurácia, precisão, revocação e F1.

Discutir qual modelo foi melhor e por quê, quais erros aparecem, quais as limitações e o que poderia melhorar. Depois preencher a tabela de resultados no README.

In [ ]:
# Pessoa C
# avaliar o modelo final no teste e gerar as métricas

Discussão:

## Uso de ferramentas de IA

- Ferramenta: Claude (assistente de IA da Anthropic).
- Finalidade: dividir as tarefas entre os integrantes, montar a estrutura do notebook e do README e apoiar a redação dos textos das seções.
- Parte do trabalho: planejamento, organização dos arquivos e redação dos textos.

Acrescentar outras ferramentas de IA aqui, caso sejam usadas em outras partes.